In [1]:
print("a")

a


In [1]:
from pathlib import Path
import requests

from pystac_client import Client
import planetary_computer

OUT_DIR = Path("../datasets/sentinel2_visual")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLOUD_DIR = OUT_DIR.parent / "cloud_overlays_png_640"
CLOUD_DIR.mkdir(parents=True, exist_ok=True)

# 画像のダウンロード

In [ ]:
import time

# 既存の OUT_DIR / requests / Client / planetary_computer を利用
# 必要なら catalog を再作成（このセル単体実行でも動くように）
if "catalog" not in globals():
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )

regions: dict[str, list[float]] = {
    # Asia-Pacific
    "tokyo_bay_jp": [139.5, 35.2, 140.2, 35.9],
    "osaka_bay_jp": [134.9, 34.3, 135.6, 34.9],
    "singapore_strait_sg": [103.4, 1.0, 104.3, 1.6],
    "manila_bay_ph": [120.6, 14.2, 121.1, 14.9],
    "jakarta_bay_id": [106.5, -6.3, 107.2, -5.8],
    "sydney_au": [150.8, -34.2, 151.5, -33.5],
    "auckland_nz": [174.4, -37.2, 175.1, -36.6],

    # Europe
    "north_sea_nl": [3.2, 51.5, 5.2, 53.0],
    "english_channel_uk_fr": [-2.5, 49.5, 1.5, 51.2],
    "mediterranean_it": [12.0, 40.0, 15.0, 42.0],
    "aegean_gr": [23.0, 36.5, 26.5, 39.5],
    "norwegian_coast_no": [4.0, 58.0, 8.5, 62.0],

    # Africa / Middle East
    "gulf_of_guinea_ng": [2.5, 4.0, 8.5, 6.8],
    "cape_town_za": [17.8, -34.5, 19.2, -33.3],
    "red_sea_eg_sa": [34.0, 21.0, 39.5, 27.5],
    "persian_gulf_ae_ir": [52.0, 24.0, 56.5, 27.8],

    # Americas
    "new_york_us": [-74.4, 40.3, -73.4, 41.1],
    "san_francisco_us": [-123.0, 37.0, -121.5, 38.3],
    "gulf_of_mexico_us": [-91.5, 27.0, -88.0, 30.0],
    "caribbean_pa": [-80.5, 8.4, -78.5, 10.2],
    "rio_brazil_br": [-44.0, -23.6, -42.8, -22.6],
    "chile_coast_cl": [-73.5, -34.5, -71.0, -32.0],

    # Open ocean (coastline以外も確保)
    "north_pacific_open_ocean": [-160.0, 20.0, -150.0, 30.0],
    "south_indian_open_ocean": [70.0, -35.0, 85.0, -25.0],
}

date_range = "2024-01-01/2025-12-31"
cloud_lt = 20 # クラウドカバー率の上限（%）
max_items_per_region = 2 # 各リージョンから最大何アイテムまでダウンロードするか（Pass数）
sleep_sec = 30  # ダウンロード間のウェイト（お行儀よく）

existing = {p.name for p in OUT_DIR.glob("*_visual.tif")}
downloaded = 0
skipped = 0

# --- Phase 1: 全リージョンの候補アイテムを収集 ---
region_queues: dict[str, list] = {}
for region_name, bbox in regions.items():
    print(f"[{region_name}] searching...")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=date_range,
        query={"eo:cloud_cover": {"lt": cloud_lt}},
    )
    items = [item for item in search.items() if item.assets.get("visual") is not None]
    region_queues[region_name] = items
    print(f"  found: {len(items)} items")

# --- Phase 2: ラウンドロビンでダウンロード（リージョン網羅を優先）---
for pass_idx in range(max_items_per_region):
    for region_name, items in region_queues.items():
        if pass_idx >= len(items):
            continue

        item = items[pass_idx]
        asset = item.assets["visual"]
        out = OUT_DIR / f"{item.id}_visual.tif"

        if out.name in existing or out.exists():
            skipped += 1
            continue

        try:
            with requests.get(asset.href, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(out, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            existing.add(out.name)
            downloaded += 1
            print(f"  [{region_name}] pass {pass_idx + 1}/{max_items_per_region}: {out.name}")
            time.sleep(sleep_sec)
        except Exception as e:
            print(f"  [{region_name}] failed: {item.id} ({e})")

print(f"\nDone. downloaded={downloaded}, skipped={skipped}, total_files={len(existing)}")


# 雲の切り出し

In [ ]:
import numpy as np
import rasterio
from PIL import Image, ImageFilter

if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR is not defined. Run the setup cell first.")

TILE_SIZE = 640  # クラウドタイルのサイズ（Sentinel-2の10m解像度で640px=6.4km四方）
STRIDE = 640     # タイルの抽出間隔
MIN_CLOUD_RATIO = 0.30  # タイル内のクラウドピクセルの最小割合
MAX_CLOUD_RATIO = 1.0   # タイル内のクラウドピクセルの最大割合
MIN_VALID_RATIO = 0.95  # タイル内の有効ピクセルの最小割合
MIN_ALPHA_MEAN = 100    # タイル内のアルファ平均の最小値（薄い雲・ヘイズを除外）
MAX_TILES_PER_TIF = 1000  # 各TIFから抽出する最大タイル数（スコア上位から）


def _to_float_hwc(rgb_raw: np.ndarray) -> np.ndarray:
    """Convert raw integer CHW array to float32 HWC in [0, 1]."""
    if rgb_raw.dtype == np.uint8:
        scale = 255.0
    elif rgb_raw.dtype == np.uint16:
        scale = 10000.0 if int(rgb_raw.max()) > 255 else 255.0
    else:
        scale = float(np.iinfo(rgb_raw.dtype).max)
    return np.clip(np.moveaxis(rgb_raw, 0, -1).astype(np.float32) / scale, 0, 1)


def _stretch_to_uint8(rgb: np.ndarray) -> np.ndarray:
    """Convert 3-band raw CHW image to uint8 with percentile stretch (for saving only)."""
    out = np.zeros_like(rgb, dtype=np.uint8)
    for i in range(3):
        band = rgb[i].astype(np.float32)
        valid = np.isfinite(band) & (band > 0)
        if not valid.any():
            continue
        lo, hi = np.percentile(band[valid], [2, 98])
        if hi <= lo:
            hi = lo + 1
        scaled = np.clip((band - lo) / (hi - lo), 0, 1)
        out[i] = (scaled * 255).astype(np.uint8)
    return out


def _compute_cloud_alpha(rgb_f32: np.ndarray) -> np.ndarray:
    """
    Build a hard alpha mask tuned for clearly visible clouds only.
    Thin haze / cirrus are intentionally excluded — callers can adjust
    opacity at blend time to handle varying cloud thickness.
    rgb_f32: float32 HWC in [0, 1], fixed reflectance scale (not stretched).
    """
    vmax = np.max(rgb_f32, axis=2)
    vmin = np.min(rgb_f32, axis=2)
    brightness = rgb_f32.mean(axis=2)
    saturation = (vmax - vmin) / (vmax + 1e-6)
    whiteness = 1.0 - saturation

    # はっきりした雲のみ: 高輝度 + 高无彩色度（閾値を厳格化）
    cloud_core = (brightness > 0.80) & (whiteness > 0.65)
    cloud_edge = (brightness > 0.72) & (whiteness > 0.58)

    base_alpha = np.where(cloud_core, 1.0, np.where(cloud_edge, 0.60, 0.0))
    detail_alpha = (
        np.clip((brightness - 0.72) / 0.28, 0, 1)
        * np.clip((whiteness - 0.55) / 0.45, 0, 1)
    )
    alpha = np.maximum(base_alpha, detail_alpha)

    alpha_img = Image.fromarray((alpha * 255).astype(np.uint8))
    alpha_img = alpha_img.filter(ImageFilter.MaxFilter(5))
    alpha_img = alpha_img.filter(ImageFilter.GaussianBlur(radius=8))
    return np.array(alpha_img, dtype=np.uint8)


saved = 0
skipped = 0

for tif_path in sorted(OUT_DIR.glob("*.tif")):
    try:
        with rasterio.open(tif_path) as src:
            if src.count < 3:
                print(f"[skip] {tif_path.name}: needs at least 3 bands")
                skipped += 1
                continue
            rgb_raw = src.read([1, 2, 3])
    except Exception as e:
        print(f"[skip] {tif_path.name}: read failed ({e})")
        skipped += 1
        continue

    rgb_f32 = _to_float_hwc(rgb_raw)
    rgb_u8_hwc = np.moveaxis(_stretch_to_uint8(rgb_raw), 0, -1)

    h, w = rgb_f32.shape[:2]
    if h < TILE_SIZE or w < TILE_SIZE:
        print(f"[skip] {tif_path.name}: smaller than {TILE_SIZE}px ({h}x{w})")
        skipped += 1
        continue

    candidates: list[tuple[float, int, int, float]] = []

    for y in range(0, h - TILE_SIZE + 1, STRIDE):
        for x in range(0, w - TILE_SIZE + 1, STRIDE):
            tile_f32 = rgb_f32[y:y + TILE_SIZE, x:x + TILE_SIZE]

            valid_ratio = float((tile_f32.mean(axis=2) > 0.01).mean())
            if valid_ratio < MIN_VALID_RATIO:
                continue

            alpha = _compute_cloud_alpha(tile_f32)

            # アルファ平均が低い = 薄い雲・ヘイズ → 除外
            if float(alpha.mean()) < MIN_ALPHA_MEAN:
                continue

            cloud_ratio = float((alpha > 20).mean())
            if not (MIN_CLOUD_RATIO <= cloud_ratio <= MAX_CLOUD_RATIO):
                continue

            score = (1.0 - abs(cloud_ratio - 0.50)) + float(alpha.mean()) / 255.0
            candidates.append((score, x, y, cloud_ratio))

    if not candidates:
        print(f"[skip] {tif_path.name}: no usable cloud tiles")
        skipped += 1
        continue

    candidates.sort(reverse=True)
    used = 0

    for _, x, y, cloud_ratio in candidates[:MAX_TILES_PER_TIF]:
        out_path = CLOUD_DIR / f"{tif_path.stem}__y{y}_x{x}.png"
        if out_path.exists():
            continue

        tile_f32 = rgb_f32[y:y + TILE_SIZE, x:x + TILE_SIZE]
        alpha = _compute_cloud_alpha(tile_f32)

        tile_u8 = rgb_u8_hwc[y:y + TILE_SIZE, x:x + TILE_SIZE]
        rgba = np.dstack([tile_u8, alpha])
        Image.fromarray(rgba, mode="RGBA").save(out_path)

        used += 1
        saved += 1
        print(f"[save] {out_path.name} cloud_ratio={cloud_ratio:.3f}")

    if used == 0:
        skipped += 1

print(f"\nDone. saved={saved}, skipped={skipped}, out_dir={CLOUD_DIR}")


## 確認

In [ ]:
from pathlib import Path
import cv2
import numpy as np
from medetect.yolo.augment import RandomCloudOverlay
from PIL import Image

# ダミー画像を生成（または実際の学習画像を使用）
img = np.full((640, 640, 3), 100, dtype=np.uint8)

# Augmentation を作成・適用
augment = RandomCloudOverlay(
    cloud_dir=CLOUD_DIR,
    alpha_range=(0.3, 0.6),
    scale_range=(0.6, 1.2),
    p=1.0  # 確実に適用
)

result = augment(image=img)["image"]

# 結果を保存
# cv2.imwrite("result.png", result)
display(Image.fromarray(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)))